`EXTRACT` — это функция в `PostgreSQL`, которая "вытаскивает" одну составную часть из значения даты/времени.

`СИНТАКСИС`

```sql
EXTRACT(field FROM source)
```
- `field` -  какую часть извлечь (год, месяц, день, час, минута, секунда...)
- `source` — значение типа date, timestamp, time, interval

ВОЗВРАЩАЕТ `numeric`  (число с плавающей точкой).

`Примеры`

```sql
SELECT EXTRACT(YEAR  FROM '2023-05-15'::date);   -- 2023----  year [jə:]
SELECT EXTRACT(MONTH FROM '2023-05-15'::date);   -- 5-------  month [mʌnθ] 
SELECT EXTRACT(DAY   FROM '2023-05-15'::date);   -- 15
SELECT EXTRACT(DOW   FROM '2023-05-15'::date);   -- 1 (Пн; 0 = воскресенье)
SELECT EXTRACT(DOY   FROM '2023-05-15'::date);   -- 135 (день в году)
SELECT EXTRACT(EPOCH FROM '2023-05-15'::date);   -- секунд с 1970-01-01
```

`Ключевые поля`

- `YEAR`, `MONTH`, `DAY` ------------ год, месяц, день
- `HOUR`, `MINUTE`, `SECOND`--------- час, минута, секунда
- `DOW`	----------------------------- день недели (0=воскресенье ... 6=суббота)
- `DOY` ----------------------------- день в году (1-366)
- `DOW vs ISODOW` ------------------- `ISODOW` — 1=понедельник ... 7=воскресенье
- `WEEK (ISO)` ---------------------- номер недели в году
- `QUARTER` ------------------------- квартал (1-4)
- `EPOCH` --------------------------- число секунд от 1970-01-01|

`В контексте вашего запроса`

```sql
WHERE EXTRACT(MONTH FROM o.order_date) = 1
```

- Для каждой строки `order_date` (например 2023-05-15) берётся месяц → 5

- Сравнивается с 1 (январь)

- Если равно — строка попадает в выборку

- Именно поэтому достигается "независимость от года": мы отбрасываем весь год и смотрим только на номер месяца. Заказы 2023-01-... и 2025-01-... оба пройдут фильтр.

`Тонкости и ограничения`

- Возвращает `numeric`, поэтому сравнение с целым 1 работает корректно, но для строгого типа лучше = 1 (не = 1.0) — не критично, PG неявно приведёт.

- Для больших таблиц быстрее будет фильтр диапазоном:

```sql
WHERE order_date >= make_date(2000 + EXTRACT(YEAR FROM order_date)::int, 1, 1)
```

или проще — по конкретному месяцу:

```sql
WHERE (order_date - date_trunc('year', order_date)) BETWEEN
      date '0001-01-01' AND date '0001-01-31'
```
